# AI Skin Type Classification with EfficientNetB0
This notebook trains a skin type classifier using TensorFlow.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model


In [2]:
train_dir = r"C:\Users\prave\OneDrive\Desktop\jupyter notebook 333\Skin Type Identification Research\train"
test_dir = r"C:\Users\prave\OneDrive\Desktop\jupyter notebook 333\Skin Type Identification Research\test"

IMG_SIZE=(224,224)
BATCH_SIZE=32


In [3]:
train_ds=tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

val_ds=tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

test_ds=tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE)

class_names=train_ds.class_names
print(class_names)


Found 1800 files belonging to 5 classes.
Using 1440 files for training.
Found 1800 files belonging to 5 classes.
Using 360 files for validation.
Found 225 files belonging to 5 classes.
['Combination', 'Dry', 'Normal', 'Oily', 'Sensitive']


In [4]:
normalization=layers.Rescaling(1./255)

train_ds=train_ds.map(lambda x,y:(normalization(x),y))
val_ds=val_ds.map(lambda x,y:(normalization(x),y))
test_ds=test_ds.map(lambda x,y:(normalization(x),y))

AUTOTUNE=tf.data.AUTOTUNE
train_ds=train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds=val_ds.cache().prefetch(AUTOTUNE)
test_ds=test_ds.cache().prefetch(AUTOTUNE)


In [5]:
data_augmentation=tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2)
])


In [6]:
base_model=EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable=False

inputs=tf.keras.Input(shape=(224,224,3))
x=data_augmentation(inputs)
x=base_model(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dropout(0.3)(x)
x=layers.Dense(256,activation="relu")(x)
x=layers.Dropout(0.3)(x)
outputs=layers.Dense(len(class_names),activation="softmax")(x)

model=Model(inputs,outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │           1,285 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,378,792 (16.70 MB)

 Trainable params: 329,221 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
history=model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)


In [ ]:
loss,accuracy=model.evaluate(test_ds)
print("Test Accuracy:",accuracy*100)

model.save("skin_type_classifier.keras")


In [ ]:
plt.plot(history.history["accuracy"],label="Train")
plt.plot(history.history["val_accuracy"],label="Validation")
plt.legend()
plt.show()

plt.plot(history.history["loss"],label="Train Loss")
plt.plot(history.history["val_loss"],label="Validation Loss")
plt.legend()
plt.show()
